# Formato

Con Agro descargaremos los archivos que necesitemos para nuestro analisis. Sin embargo, debemos asegurarnos de no perder información al momento de descargar la información. No confundir con los datos faltantes. Estos se originaron por la naturaleza de la fuente. Nuestro problema es con respecto a información que si estaba pero perdimos al momento de cargar la información.

## Agro en acción

Denominando Agro a nuestra herramienta para la descarga de archivos del DGSIAP. Haremos una prueba.

Supongamos que queremos la información de la producción agricola municipal de todos los años disponibles

In [1]:
# Importamos el diccionario maestro. (No olvide elegir su categoria)
from scripts import dgsiap

# Importamos las herramientas para descarga
from scripts import download as d 

# Informacion municipal historica
request = {
    "category":"Municipal",
    "period": "*"}

client = dgsiap.dict_master()

# Generamos nuestra solicitud
dic_link = d.downassist(request=request, ref=client)

# Descargamos los archivos requeridos
d.downbot(dic_link)

Bienvenido al script de descarga de datos abiertos del DGSIAP.
Conexion Establecida
Usted escogio Agricola
Los datos historicos se presentan en el espacio denomiado: Información por nivel
Los datos se encuentran divididos en 3 bloques:

1. Municipal
2. Nacional
3. No seguimiento
El archivo ya fue descargado. Se encuentra en /data/external/Diccionario_agricola_2003_a_2025.xlsx
El archivo ya fue descargado. Se encuentra en /data/external/Cierre_agricola_mun_2025.csv
El archivo ya fue descargado. Se encuentra en /data/external/Cierre_agricola_mun_2024.csv
El archivo ya fue descargado. Se encuentra en /data/external/Cierre_agricola_mun_2023.csv
El archivo ya fue descargado. Se encuentra en /data/external/Cierre_agricola_mun_2022.csv
El archivo ya fue descargado. Se encuentra en /data/external/Cierre_agricola_mun_2021.csv
El archivo ya fue descargado. Se encuentra en /data/external/Cierre_agricola_mun_2020.csv
El archivo ya fue descargado. Se encuentra en /data/external/Cierre_agricola_mun_

## Encoding.

Uno de los problemas más conocidos al momento de descargar información es el desconocimiento del enconding usado al generar el conjunto de datos. Por defecto procuraremos usar el formato estándar UTF-8. Sin embargo, es posible encontrar archivos con coficiaciones distintas. Entonces debemos buscar la forma de leerlas con la minima pérdida

In [2]:
import cchardet
import pandas as pd
from pathlib import Path
import os

# Tomamos algun archivo .csv

csv_set = list(d.DATA_PATH.glob("*.csv"))
csv = csv_set[0]

try:
    pd.read_csv(csv)
except UnicodeDecodeError as e:
    print(e)

'utf-8' codec can't decode byte 0xf1 in position 350: invalid continuation byte


Como podemos observar, ni siquiera es posible leerlo. Entonces debemos encontrar su codificación para poder leerlo. Para ello usaremos la libreria `cchardet`.

In [3]:
with open(csv, 'rb') as f: # Leemos su archivo binario asociado
    data = f.read()
    encode = cchardet.detect(data)

print(encode)

{'encoding': 'ISO-8859-1', 'confidence': 0.8167673945426941, 'language': 'es'}


Viendo que nos devuelve el encoding que la libreria detecto, además de 2 valores más

* confidence: Un valor entre 0 y 1 que define la seguridad de la respuesta.
* language: el idioma que detecto. (Este puede ser importante si nos detecta un idioma no relacionado con el documento)

Entonces aplicamos lo anterior al resto de documentos. Algo a comentar de la libreria es que es algo lenta para encontrarlos. Sin embargo, lo esperado es hacer esto durante la descarga de los archivos. Asi cuando las descargas terminen tener todo listo para el analisis.

In [11]:
from scripts.format import AUXILIAR_PATH
import json

def get_encoding(filepath):
    with open(filepath, 'rb') as file:
        data=file.read()
        res = cchardet.detect(data)
    return res

encode_guide_path = AUXILIAR_PATH / "dgsiap_encode.json"

# Verificamos que el json que albergara los encodes exista
# Si no. Lo crearemos
dgsiap_encode = {}
# Paso 1: Construir el diccionario respectivo.
if not os.path.exists(encode_guide_path):
    for c in csv_set:
        filename = str(c).split("/")[-1]
        dgsiap_encode[filename] = get_encoding(c)
    # Guardamos y creamos el .json
    with open(encode_guide_path,"w") as file:
        json.dump(dgsiap_encode, file)

Hacemos la prueba final. Intentamos leer todos los csv.

In [14]:
for c in csv_set:
    filename = str(c).split("/")[-1]
    code = dgsiap_encode[filename]
    try:
        v = pd.read_csv(c, encoding= code['encoding'], low_memory=False)
        print(filename)
        print(v.columns)
    except UnicodeDecodeError as e:
        print(filename, e)


Cierre_agricola_mun_2025.csv
Index(['Anio', 'Idestado', 'Nomestado', 'Idddr', 'Nomddr', 'Idcader',
       'Nomcader', 'Idmunicipio', 'Nommunicipio', 'Idciclo',
       'Nomcicloproductivo', 'Idmodalidad', 'Nommodalidad', 'Idunidadmedida',
       'Nomunidad', 'Idcultivo', 'Nomcultivo', 'Sembrada', 'Cosechada',
       'Siniestrada', 'Volumenproduccion', 'Rendimiento', 'Preciomediorural',
       'Valorproduccion'],
      dtype='str')
Cierre_agricola_mun_2024.csv
Index(['Anio', 'Idestado', 'Nomestado', 'Idddr', 'Nomddr', 'Idcader',
       'Nomcader', 'Idmunicipio', 'Nommunicipio', 'Idciclo',
       'Nomcicloproductivo', 'Idmodalidad', 'Nommodalidad', 'Idunidadmedida',
       'Nomunidad', 'Idcultivo', 'Nomcultivo', 'Sembrada', 'Cosechada',
       'Siniestrada', 'Volumenproduccion', 'Rendimiento', 'Preciomediorural',
       'Valorproduccion'],
      dtype='str')
Cierre_agricola_mun_2023.csv
Index(['Anio', 'Idestado', 'Nomestado', 'Idddr', 'Nomddr', 'Idcader',
       'Nomcader', 'Idmunicipio',

Entonces ya es posible leer cualquier archivo .csv mediante pandas. Sin embargo, esto no garantiza que no existan errores a futuro que debamos corregir durante el analisis. La prioridad de esta sección es que cualquier archivo pueda ser consultado para el analisis.

In [15]:
csv_sample = csv_set[0]

test = pd.read_csv(csv_sample, encoding= code['encoding'], low_memory=False)
print(test)


       Anio  Idestado       Nomestado  Idddr          Nomddr  Idcader  \
0      2025         1  Aguascalientes      1  Aguascalientes        1   
1      2025         1  Aguascalientes      1  Aguascalientes        1   
2      2025         1  Aguascalientes      1  Aguascalientes        1   
3      2025         1  Aguascalientes      1  Aguascalientes        1   
4      2025         1  Aguascalientes      1  Aguascalientes        1   
...     ...       ...             ...    ...             ...      ...   
35897  2025        32       Zacatecas    190           Jalpa        4   
35898  2025        32       Zacatecas    190           Jalpa        4   
35899  2025        32       Zacatecas    190           Jalpa        4   
35900  2025        32       Zacatecas    190           Jalpa        4   
35901  2025        32       Zacatecas    190           Jalpa        4   

             Nomcader  Idmunicipio    Nommunicipio  Idciclo  ... Nomunidad  \
0      Aguascalientes            1  Aguascali

En primera instancia se ve bien, pero no debemos bajar la guardia. Sin embargo, los detalles pueden esperar un momento. Ahora tenemos otro problema. Entender que hace cada variable del documento. Aqui es donde entran los diccionarios de datos.

## Diccionario de Datos

Usando el csv de prueba. Obtendremos el diccionario asociado.

In [16]:
print(csv_sample) # El url nos da información del diccionario de datos asociado.
print(dic_link)  # Revisamos la información solicitada

dicts = list(d.DATA_PATH.glob("*.xlsx"))
print(dicts)

# Esto se puede hacer más limpio con un diccionario de archivos descargados. 
v = dic_link['Municipal']['Codebook']
print("Link", v)  # Obtenemos el url de donde vino.
print(d.get_filename(d.parse_link(v)))
dictname = d.get_filename(d.parse_link(v))
codebook = pd.read_excel(d.DATA_PATH / dictname)

print(codebook.head(20))

/home/david/Documents/Repositories/academica/proyecto_agronomia/scripts/../data/external/Cierre_agricola_mun_2025.csv
{'Municipal': {'Codebook': '/index.php?view=5D24F995-E734D711-1EA6F8D2-9D1387D7', 'Data': {'2025': URL('/index.php?view=10AE434F-A2158368-A120BC5A-EDF4AFAA&ANIO=2025'), '2024': URL('/index.php?view=10AE434F-A2158368-A120BC5A-EDF4AFAA&ANIO=2024'), '2023': URL('/index.php?view=10AE434F-A2158368-A120BC5A-EDF4AFAA&ANIO=2023'), '2022': URL('/index.php?view=10AE434F-A2158368-A120BC5A-EDF4AFAA&ANIO=2022'), '2021': URL('/index.php?view=10AE434F-A2158368-A120BC5A-EDF4AFAA&ANIO=2021'), '2020': URL('/index.php?view=10AE434F-A2158368-A120BC5A-EDF4AFAA&ANIO=2020'), '2019': URL('/index.php?view=10AE434F-A2158368-A120BC5A-EDF4AFAA&ANIO=2019'), '2018': URL('/index.php?view=10AE434F-A2158368-A120BC5A-EDF4AFAA&ANIO=2018'), '2017': URL('/index.php?view=10AE434F-A2158368-A120BC5A-EDF4AFAA&ANIO=2017'), '2016': URL('/index.php?view=10AE434F-A2158368-A120BC5A-EDF4AFAA&ANIO=2016'), '2015': URL

Ahora usaremos el diccionario de prueba para adaptar el formato del excel.

In [17]:
content = codebook.iloc[8:].reset_index(drop=True)
content.columns = content.iloc[0]
content = content.iloc[1:]
content.set_index(content.columns[1], inplace=True)
content = content.drop(columns="Consecutivo\nen la base")
print(content)

w = codebook.iloc[33:38, :1].reset_index(drop=True)
w = w.iloc[1:]
w.columns = ['Variable']
values = [v.split(":") for v in w['Variable']]
code_index = [l[0] for l in values]
col_vals = [l[1].strip("") for l in values]
w.index = code_index
w['Variable'] = col_vals

print(w)

0                  Tipo Long.  Rangos\nválidos  \
Nombre\nde campo                                 
Anio                  N     4           {CCCC}   
Idestado              N     2        {1,…,32 }   
Nomestado             T    60            {CCC}   
Idddr                 N     3        {1,…,193}   
Nomddr                T    60            {CCC}   
Idcader               N     2         {1,…,19}   
Nomcader              T    90            {CCC}   
Idmunicipio           T     3       {1,….,570}   
Nommunicipio          T    45            {CCC}   
Idciclo               N     1         {1,….,3}   
Nomcicloproductivo    T    40            {CCC}   
Idmodalidad           N     1            {1,2}   
Nommodalidad          T    20            {CCC}   
Idunidadmedida        N     1         {1,….,6}   
Nomunidad             T    40            {CCC}   
Idcultivo             N     3       {1,….,410}   
Nomcultivo            T    40            {CCC}   
Sembrada              D    10    {########.##}   


Elimiando columnas innecesarias y extrayendo la simbologia de los tipos de datos obtenemos un formato adecuado para consulta durante el analisis 

In [18]:
def format_codebook(code):
    content = code.iloc[8:33].reset_index(drop=True)
    content.columns = content.iloc[0]
    content = content.iloc[1:]
    content.set_index(content.columns[1], inplace=True)
    content = content.drop(columns="Consecutivo\nen la base")

    w = code.iloc[34:38, :1]
    w.columns = ['Variable']
    values = [v.split(":") for v in w['Variable']]
    code_index = [l[0] for l in values]
    col_vals = [l[1].strip("") for l in values]
    w.index = code_index
    w['Variable'] = col_vals
    return content, w


format_codebook(codebook)[0]

,Tipo,Long.,Rangos\nválidos,Descripción,Observaciones
Nombre de campo,,,,,
Anio,N,4,{CCCC},Año de la información,Corresponde a la referencia temporal de los da...
Idestado,N,2,"{1,…,32 }",Código que identifica a la entidad federativa ...,Clave geoestadística consecutiva que se asigna...
Nomestado,T,60,{CCC},Nombre oficial de la entidad federativa o estado,La entidad federativa es la unidad geográfica ...
Idddr,N,3,"{1,…,193}",Código que identifica al Distrito de Desarrol...,Clave interna que utiliza la Secretaría de Agr...
Nomddr,T,60,{CCC},Nombre que identifica al Distrito de Desarrol...,Nombre que utiliza la Secretaría de Agricultur...
Idcader,N,2,"{1,…,19}",Código que identifica al Centro de Apoyo al D...,Clave interna que utiliza la Secretaría de Agr...
Nomcader,T,90,{CCC},Nombre que identifica al Centro de Apoyo al D...,Nombre que utiliza la Secretaría de Agricultur...
Idmunicipio,T,3,"{1,….,570}",Código que identifica al municipio o delegació...,Unidad geográfica de la División territorial p...
Nommunicipio,T,45,{CCC},Nombre del municipio o delegación de producción,El municipio refiere a la unidad geográfica de...
